In [ ]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.special import gammaln, gamma as gamma_func
from scipy.optimize import minimize
from scipy.stats import (gamma as gamma_dist, weibull_min,
                         lognorm as lognorm_dist, expon as expon_dist, kstest) # Import kstest
import warnings
import os # Import the os module
warnings.filterwarnings('ignore')

# ============================================================
# DATA DEATH BENEFIT (40 observasi)
# ============================================================

data = np.array([
    148104, 24444, 87420, 68460, 115200, 96600, 213792, 32512, 185856, 224928,
    146718, 221001, 192984, 279510, 226270, 177320, 113212, 245234, 197780, 250228,
    281160, 204204, 10248, 62160, 69696, 146880, 146520, 263736, 302202, 248292,
    265356, 212652, 242136, 465408, 630168, 350208, 351648, 273840, 738234, 205632
], dtype=float)

n = len(data)
x = data.copy()

# Normalisasi skala untuk stabilitas numerik (krusial untuk Hessian)
# Semua estimasi FIC dilakukan pada x_sc, lalu dikembalikan ke skala asli
SCALE = x.mean()
x_sc = x / SCALE

# ============================================================
# 1. STATISTIKA DESKRIPTIF
# ============================================================

print("=" * 65)
print("         STATISTIKA DESKRIPTIF DEATH BENEFIT")
print("=" * 65)
print(f"  n (observasi)             : {n}")
print(f"  Mean                      : {x.mean():,.2f}")
print(f"  Median                    : {np.median(x):,.2f}")
print(f"  Modus (approx)            : {pd.Series(x).mode().values[0]:,.0f}")
print(f"  Variansi                  : {x.var(ddof=1):,.2f}")
print(f"  Standar Deviasi           : {x.std(ddof=1):,.2f}")
print(f"  Koefisien Variasi (CV)    : {x.std(ddof=1)/x.mean()*100:.2f}%")
print(f"  Minimum                   : {x.min():,.0f}")
print(f"  Maksimum                  : {x.max():,.0f}")
print(f"  Range                     : {x.max()-x.min():,.0f}")
print(f"  Q1 (P25)                  : {np.percentile(x, 25):,.2f}")
print(f"  Q2 (P50)                  : {np.percentile(x, 50):,.2f}")
print(f"  Q3 (P75)                  : {np.percentile(x, 75):,.2f}")
print(f"  IQR                       : {np.percentile(x,75)-np.percentile(x,25):,.2f}")
print(f"  Skewness                  : {pd.Series(x).skew():.4f}")
print(f"  Kurtosis (excess)         : {pd.Series(x).kurt():.4f}")
print(f"  Kuantil 90%               : {np.percentile(x, 90):,.2f}")
print(f"  Kuantil 95%               : {np.percentile(x, 95):,.2f}")
print(f"  Kuantil 99%               : {np.percentile(x, 99):,.2f}")
print()

# ============================================================
# 2. GRAFIK: HISTOGRAM + DD-PLOT
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax1 = axes[0]
ax1.hist(x, bins=10, density=True, color='steelblue', edgecolor='white',
         linewidth=0.8, alpha=0.7, label='Observasi')
ax1.set_title('Histogram Death Benefit', fontsize=12, fontweight='bold')
ax1.set_xlabel('Death Benefit (Rp)')
ax1.set_ylabel('Densitas')
ax1.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v/1e6:.1f}M'))
ax1.grid(True, alpha=0.3)

ax2 = axes[1]

# ============================================================
# 3. ESTIMASI PARAMETER MLE (pada data asli)
# ============================================================

# --- (a) GAMMA: f(x) = (beta^alpha/Gamma(alpha)) * x^(alpha-1) * exp(-beta*x)
#     beta = rate = 1/scale, sesuai PDF di gambar

def negloglik_gamma(params):
    alpha, beta = params
    if alpha <= 0 or beta <= 0:
        return np.inf
    ll = (n * (alpha * np.log(beta) - gammaln(alpha))
          + (alpha - 1) * np.sum(np.log(x))
          - beta * np.sum(x))
    return -ll

x_bar = x.mean()
s2 = x.var(ddof=1)
alpha_init = x_bar**2 / s2
beta_init  = x_bar / s2

res_gamma = minimize(negloglik_gamma, [alpha_init, beta_init],
                     method='Nelder-Mead',
                     options={'xatol':1e-10,'fatol':1e-10,'maxiter':50000})
alpha_mle, beta_mle = res_gamma.x
ll_gamma  = -res_gamma.fun
k_gamma   = 2

# --- (b) WEIBULL: f(x) = (k/lam)*(x/lam)^(k-1)*exp(-(x/lam)^k)

def negloglik_weibull(params):
    k, lam = params
    if k <= 0 or lam <= 0:
        return np.inf
    ll = (n * np.log(k) - n * k * np.log(lam)
          + (k - 1) * np.sum(np.log(x))
          - np.sum((x / lam)**k))
    return -ll

k_init  = (x.std(ddof=1) / x.mean())**(-1.086)
lam_init = x.mean() / gamma_func(1 + 1/k_init)

res_wb = minimize(negloglik_weibull, [k_init, lam_init],
                  method='Nelder-Mead',
                  options={'xatol':1e-10,'fatol':1e-10,'maxiter':50000})
k_wb_mle, lam_wb_mle = res_wb.x
ll_weibull = -res_wb.fun
k_weibull  = 2

# --- (c) LOGNORMAL: MLE analitik

log_x = np.log(x)
mu_ln_mle    = log_x.mean()
sigma_ln_mle = log_x.std(ddof=0)

def negloglik_lognorm(params):
    mu, sigma = params
    if sigma <= 0:
        return np.inf
    ll = (-n * np.log(sigma) - n * 0.5 * np.log(2 * np.pi)
          - np.sum(np.log(x))
          - np.sum((log_x - mu)**2) / (2 * sigma**2))
    return -ll

ll_lognorm = -negloglik_lognorm([mu_ln_mle, sigma_ln_mle])
k_lognorm  = 2

# --- (d) EKSPONENSIAL: MLE analitik

lambda_exp_mle = 1.0 / x.mean()
ll_exp = n * np.log(lambda_exp_mle) - lambda_exp_mle * np.sum(x)
k_exp  = 1

print("=" * 65)
print("         ESTIMASI PARAMETER MLE")
print("=" * 65)
print()
print("  [1] Distribusi GAMMA (sesuai PDF gambar: beta = rate)")
print(f"      alpha (shape) = {alpha_mle:.6f}")
print(f"      beta  (rate)  = {beta_mle:.6e}")
print(f"      E[X] teoritis = {alpha_mle/beta_mle:,.2f}")
print(f"      Log-Likelihood = {ll_gamma:.4f}")
print()
print("  [2] Distribusi WEIBULL")
print(f"      k      (shape) = {k_wb_mle:.6f}")
print(f"      lambda (scale) = {lam_wb_mle:,.4f}")
print(f"      Log-Likelihood = {ll_weibull:.4f}")
print()
print("  [3] Distribusi LOGNORMAL")
print(f"      mu    = {mu_ln_mle:.6f}")
print(f"      sigma = {sigma_ln_mle:.6f}")
print(f"      Log-Likelihood = {ll_lognorm:.4f}")
print()
print("  [4] Distribusi EKSPONENSIAL")
print(f"      lambda = {lambda_exp_mle:.6e}")
print(f"      E[X] teoritis = {1/lambda_exp_mle:,.2f}")
print(f"      Log-Likelihood = {ll_exp:.4f}")
print()

# ============================================================
# 4. KOLMOGOROV-SMIRNOV
# ============================================================

# Menggunakan scipy.stats.kstest untuk mendapatkan p-value
# Perlu menentukan cdf_name untuk kstest

# KSTEST untuk Gamma
# Parameter untuk scipy.stats.gamma: a (shape), loc, scale = 1/beta (rate)
ks_stat_gamma, ks_pvalue_gamma = kstest(x, 'gamma', args=(alpha_mle, 0, 1/beta_mle))

# KSTEST untuk Weibull
# Parameter untuk scipy.stats.weibull_min: c (shape), loc, scale
ks_stat_weibull, ks_pvalue_weibull = kstest(x, 'weibull_min', args=(k_wb_mle, 0, lam_wb_mle))

# KSTEST untuk Lognormal
# Parameter untuk scipy.stats.lognorm: s (shape), loc, scale = exp(mu)
ks_stat_lognorm, ks_pvalue_lognorm = kstest(x, 'lognorm', args=(sigma_ln_mle, 0, np.exp(mu_ln_mle)))

# KSTEST untuk Eksponensial
# Parameter untuk scipy.stats.expon: loc, scale = 1/lambda
ks_stat_exp, ks_pvalue_exp = kstest(x, 'expon', args=(0, 1/lambda_exp_mle))

alpha_level = 0.05 # Tingkat signifikansi

print("=" * 65)
print("         UJI KOLMOGOROV-SMIRNOV")
print("=" * 65)
print(f"  Tingkat Signifikansi (alpha) = {alpha_level:.2f}")
print()
print("  Catatan: Karena parameter diestimasi dari data yang sama")
print("  (bukan hipotesis a priori), p-value dari uji KS standar")
print("  bersifat konservatif. Gunakan sebagai indikasi.")
print()
print(f"  {'Distribusi':<22} {'D statistik':>12}  {'p-value':>10}  {'Kesimpulan'}")
print("  " + "-" * 65)

ks_results = [
    ('Gamma', ks_stat_gamma, ks_pvalue_gamma),
    ('Weibull', ks_stat_weibull, ks_pvalue_weibull),
    ('Lognormal', ks_stat_lognorm, ks_pvalue_lognorm),
    ('Eksponensial', ks_stat_exp, ks_pvalue_exp)
]

for label, D_stat, p_value in ks_results:
    ket = "Gagal Tolak H0 ✓" if p_value > alpha_level else "TOLAK H0 (signifikan)"
    print(f"  {label:<22} {D_stat:>12.4f}  {p_value:>10.4f}  {ket}")
print()


# ============================================================
# 5. AIC DAN BIC
# ============================================================

def aic(ll, k):
    return 2 * k - 2 * ll

def bic(ll, k, n):
    return k * np.log(n) - 2 * ll

def aicc(ll, k, n):
    a = aic(ll, k)
    return a + (2 * k * (k + 1)) / (n - k - 1)

results = {
    'Gamma':        {'ll': ll_gamma,   'k': k_gamma, 'ks_d': ks_stat_gamma, 'ks_p': ks_pvalue_gamma},
    'Weibull':      {'ll': ll_weibull, 'k': k_weibull, 'ks_d': ks_stat_weibull, 'ks_p': ks_pvalue_weibull},
    'Lognormal':    {'ll': ll_lognorm, 'k': k_lognorm, 'ks_d': ks_stat_lognorm, 'ks_p': ks_pvalue_lognorm},
    'Eksponensial': {'ll': ll_exp,     'k': k_exp, 'ks_d': ks_stat_exp, 'ks_p': ks_pvalue_exp},
}

print("=" * 65)
print("         AIC DAN BIC")
print("=" * 65)
print()
print(f"  {'Distribusi':<15} {'k':>3}  {'Log-L':>10}  {'AIC':>10}  {'AICc':>10}  {'BIC':>10}")
print("  " + "-" * 60)
for name, v in results.items():
    a  = aic(v['ll'], v['k'])
    ac = aicc(v['ll'], v['k'], n)
    b  = bic(v['ll'], v['k'], n)
    v['aic'] = a; v['aicc'] = ac; v['bic'] = b
    print(f"  {name:<15} {v['k']:>3}  {v['ll']:>10.4f}  {a:>10.4f}  {ac:>10.4f}  {b:>10.4f}")

best_aic  = min(results, key=lambda nm: results[nm]['aic'])
best_aicc = min(results, key=lambda nm: results[nm]['aicc'])
best_bic  = min(results, key=lambda nm: results[nm]['bic'])
print()
print(f"  Model terbaik (AIC  terkecil) : {best_aic}")
print(f"  Model terbaik (AICc terkecil) : {best_aicc}")
print(f"  Model terbaik (BIC  terkecil) : {best_bic}")
print()

# ============================================================
# 6. FIC — CLAESKENS & HJORT (2003)
# ============================================================
# Fokus: mu = Q_alpha(X) = kuantil ke-alpha (Value at Risk)
# FIC_j = b²_j,c + (v_pm,j / n)
#
# Komponen:
#   mu_np  : kuantil nonparametrik (empiris) sebagai "full model"
#   mu_pm  : kuantil parametrik dari distribusi j
#   g_hat  : densitas KDE di titik mu_np (bandwidth Silverman)
#   IF     : influence function = (alpha - I(x <= mu_np)) / g_hat
#   v_np   : variansi IF (variansi estimasi nonparametrik)
#   J      : matriks Fisher information (negatif Hessian / n)
#   K      : matriks kovarians outer-product skor
#   c      : gradien kuantil parametrik terhadap parameter (via num. diff.)
#   v_pm   : variansi asymptotik estimator parametrik = c' J^{-1} K J^{-1} c
#   d      : kovarians IF dengan skor
#   b²_c   : bias kuadrat terkoreksi = max(bias² - koreksi, 0)
#   FIC    : b²_c + v_pm/n
#   AFIC   : rata-rata FIC atas alpha ∈ [0.90, 0.99]
#
# Referensi: Claeskens G. & Hjort N.L. (2003). The Focused
#            Information Criterion. JASA 98(464), 900-916.
# ============================================================

print("=" * 65)
print("         FIC — CLAESKENS & HJORT (2003)")
print("=" * 65)
print()

# -- Semua komputasi FIC dilakukan pada data yang dinormalisasi (x_sc)
# -- untuk stabilitas numerik Hessian, lalu FIC dikembalikan ke skala asli

def silverman_bw(d):
    """Bandwidth Silverman's rule of thumb untuk KDE."""
    std = np.std(d, ddof=1)
    iqr = np.percentile(d, 75) - np.percentile(d, 25)
    s   = min(std, iqr / 1.34)
    return 1.06 * s * len(d)**(-0.2)

def kde_density(x_eval, d, h):
    """Densitas KDE Gaussian di titik x_eval."""
    return np.mean(np.exp(-0.5 * ((x_eval - d) / h)**2) / (np.sqrt(2 * np.pi) * h))

def num_grad(f, theta, eps=1e-5):
    """Gradien numerik (metode beda tengah)."""
    g = np.zeros(len(theta))
    for i in range(len(theta)):
        tp, tm = theta.copy(), theta.copy()
        tp[i] += eps; tm[i] -= eps
        g[i] = (f(tp) - f(tm)) / (2 * eps)
    return g

def num_hessian(f, theta, eps=1e-4):
    """Hessian numerik (beda hingga orde-4)."""
    np_ = len(theta)
    H   = np.zeros((np_, np_))
    for i in range(np_):
        for j in range(np_):
            tpp = theta.copy(); tpp[i] += eps; tpp[j] += eps
            tpm = theta.copy(); tpm[i] += eps; tpm[j] -= eps
            tmp = theta.copy(); tmp[i] -= eps; tmp[j] += eps
            tmm = theta.copy(); tmm[i] -= eps; tmm[j] -= eps
            H[i, j] = (f(tpp) - f(tpm) - f(tmp) + f(tmm)) / (4 * eps**2)
    return H

def compute_fic(logpdf_func, ppf_func, theta, x_sc, alpha=0.95):
    """
    Hitung FIC untuk satu distribusi pada satu nilai alpha.
    Semua perhitungan pada x_sc (data ternormalisasi).
    FIC dikembalikan dalam satuan (x_sc)^2 — tidak perlu rescale
    karena FIC digunakan untuk perbandingan relatif antar distribusi.
    """
    m     = len(x_sc)
    theta = np.array(theta, dtype=float)
    h     = silverman_bw(x_sc)

    # Kuantil target
    mu_np = np.quantile(x_sc, alpha)           # nonparametrik (empiris)
    mu_pm = ppf_func(alpha, theta)              # parametrik

    # Densitas KDE & influence function
    g_hat   = max(kde_density(mu_np, x_sc, h), 1e-10)
    ind     = (x_sc <= mu_np).astype(float)
    IF_vals = (alpha - ind) / g_hat             # shape (m,)
    v_np    = np.mean(IF_vals**2)

    # Fisher information matrix J = -E[Hessian logpdf] ≈ -H_total/m
    def total_ll(t):
        vals = [logpdf_func(xi, t) for xi in x_sc]
        return sum(v for v in vals if np.isfinite(v))

    H     = num_hessian(total_ll, theta)
    J_hat = -H / m + np.eye(len(theta)) * 1e-8   # regularisasi kecil
    try:
        J_inv = np.linalg.inv(J_hat)
    except np.linalg.LinAlgError:
        J_inv = np.linalg.pinv(J_hat)

    # Outer-product skor K = E[score score']
    scores = np.array([
        num_grad(lambda t, xi=xi: logpdf_func(xi, t), theta)
        for xi in x_sc
    ])
    K_hat = np.mean([np.outer(s, s) for s in scores], axis=0)

    # Gradien kuantil parametrik c = d/dtheta Q_alpha(theta)
    c_hat = num_grad(lambda t: ppf_func(alpha, t), theta)

    # Variansi asymptotik estimator parametrik (sandwich)
    sandwich = J_inv @ K_hat @ J_inv
    v_pm     = float(c_hat @ sandwich @ c_hat)

    # Koreksi bias: d = E[IF * score]
    d_hat   = np.mean([IF_vals[i] * scores[i] for i in range(m)], axis=0)
    b_sq    = (mu_pm - mu_np)**2
    corr    = (1 / m) * (v_np + v_pm - 2 * float(c_hat @ J_inv @ d_hat))
    b_sq_c  = max(b_sq - corr, 0.0)

    fic = b_sq_c + v_pm / m

    return {
        'mu_np': mu_np * SCALE,
        'mu_pm': mu_pm * SCALE,
        'bias' : (mu_pm - mu_np) * SCALE,
        'b2c'  : b_sq_c,
        'v_pm' : v_pm,
        'v_np' : v_np,
        'fic'  : fic,
    }

def compute_afic(logpdf_func, ppf_func, theta,
                 x_sc, alphas=np.arange(0.90, 1.00, 0.01)):
    """
    AFIC = rata-rata FIC atas beberapa nilai alpha (0.90-0.99).
    Mengukur performa rata-rata distribusi di seluruh ekor atas.
    """
    fics = []
    for a in alphas:
        try:
            r = compute_fic(logpdf_func, ppf_func, theta, x_sc, a)
            fics.append(r['fic'])
        except Exception:
            pass
    return float(np.mean(fics)) if fics else np.nan

# -- Log-PDF dan PPF untuk tiap distribusi pada x_sc --

# GAMMA (alpha, beta_rate) — identik dengan MLE asli karena skala tidak mengubah shape
x_bar_sc = x_sc.mean(); s2_sc = x_sc.var(ddof=1)
ai_sc = x_bar_sc**2 / s2_sc; bi_sc = x_bar_sc / s2_sc

def _negll_gamma_sc(p):
    a, b = p
    if a <= 0 or b <= 0: return np.inf
    return -(len(x_sc) * (a * np.log(b) - gammaln(a))
             + (a - 1) * np.sum(np.log(x_sc)) - b * np.sum(x_sc))

res_gsc = minimize(_negll_gamma_sc, [ai_sc, bi_sc], method='Nelder-Mead',
                   options={'xatol':1e-12,'fatol':1e-12,'maxiter':100000})
a_sc, b_sc = res_gsc.x

def gamma_logpdf_sc(xi, t):
    # untuk scipy.stats.gamma: shape a, loc=0, scale=1/beta
    return gamma_dist.logpdf(xi, a=t[0], loc=0, scale=1/t[1])

def gamma_ppf_sc(alpha, t):
    return gamma_dist.ppf(alpha, a=t[0], scale=1/t[1])

# WEIBULL
ki_sc  = (x_sc.std(ddof=1) / x_sc.mean())**(-1.086)
li_sc  = x_sc.mean() / gamma_func(1 + 1/ki_sc)

def _negll_wb_sc(p):
    k, lam = p
    if k <= 0 or lam <= 0: return np.inf
    return -(len(x_sc)*np.log(k) - len(x_sc)*k*np.log(lam)
             + (k-1)*np.sum(np.log(x_sc)) - np.sum((x_sc/lam)**k))

res_wsc = minimize(_negll_wb_sc, [ki_sc, li_sc], method='Nelder-Mead',
                   options={'xatol':1e-12,'fatol':1e-12,'maxiter':100000})
k_sc, lam_sc = res_wsc.x

def weibull_logpdf_sc(xi, t):
    return weibull_min.logpdf(xi, c=t[0], loc=0, scale=t[1])

def weibull_ppf_sc(alpha, t):
    return weibull_min.ppf(alpha, c=t[0], scale=t[1])

# LOGNORMAL
log_xsc   = np.log(x_sc)
mu_sc     = log_xsc.mean()
sigma_sc  = log_xsc.std(ddof=0)

def lognorm_logpdf_sc(xi, t):
    return lognorm_dist.logpdf(xi, s=t[1], loc=0, scale=np.exp(t[0]))

def lognorm_ppf_sc(alpha, t):
    return lognorm_dist.ppf(alpha, s=t[1], scale=np.exp(t[0]))

# EKSPONENSIAL
lam_exp_sc = 1.0 / x_sc.mean()

def exp_logpdf_sc(xi, t):
    return expon_dist.logpdf(xi, loc=0, scale=1/t[0])

def exp_ppf_sc(alpha, t):
    return expon_dist.ppf(alpha, scale=1/t[0])

# -- Kumpulan distribusi --
dists_fic = [
    ('Gamma',        gamma_logpdf_sc,   gamma_ppf_sc,   [a_sc,  b_sc]),
    ('Weibull',      weibull_logpdf_sc, weibull_ppf_sc, [k_sc,  lam_sc]),
    ('Lognormal',    lognorm_logpdf_sc, lognorm_ppf_sc, [mu_sc, sigma_sc]),
    ('Eksponensial', exp_logpdf_sc,     exp_ppf_sc,     [lam_exp_sc]),
]

# -- Hitung FIC dan AFIC --
ALPHA_FOKUS = 0.95
mu_np_global = np.quantile(x, ALPHA_FOKUS)

print(f"  Fokus         : Kuantil ke-{ALPHA_FOKUS*100:.0f}% (Value at Risk / VaR)")
print(f"  mu_np empiris : {mu_np_global:,.2f}")
print()
print("  Teori Claeskens & Hjort (JASA 2003):")
print("  FIC  = b²_c + (v_pm/n)")
print("  AFIC = rata-rata FIC atas α ∈ [0.90, 0.99]")
print()
print(f"  {'Distribusi':<15} {'Q95% pred':>12}  {'Bias':>12}  " \
      f"{'b²_c':>10}  {'v_pm/n':>10}  {'FIC':>12}  {'AFIC':>12}")
print("  " + "-" * 90)

fic_records = {}
for name, lpdf, ppf, theta in dists_fic:
    try:
        r    = compute_fic(lpdf, ppf, theta, x_sc, ALPHA_FOKUS)
        afic = compute_afic(lpdf, ppf, theta, x_sc)
        fic_records[name] = {'fic': r['fic'], 'afic': afic, **r}
        print(f"  {name:<15} {r['mu_pm']:>12,.2f}  {r['bias']:>12,.2f}  " \
              f"{r['b2c']:>10.6f}  {r['v_pm']/n:>10.6f}  " \
              f"{r['fic']:>12.8f}  {afic:>12.8f}")
    except Exception as e:
        print(f"  {name:<15} ERROR: {e}")
        fic_records[name] = {'fic': np.inf, 'afic': np.inf}

best_fic  = min(fic_records, key=lambda nm: fic_records[nm]['fic'])
best_afic = min(fic_records, key=lambda nm: fic_records[nm]['afic'])

print()
print(f"  Model terbaik (FIC  terkecil) : {best_fic}")
print(f"  Model terbaik (AFIC terkecil) : {best_afic}")
print()
print("  Interpretasi komponen FIC:")
print("  • b²_c   : bias kuadrat terkoreksi estimasi kuantil")
print("             (selisih antara kuantil parametrik vs empiris,")
print("              setelah koreksi variansi finite-sample)")
print("  • v_pm/n : variansi asymptotik estimator parametrik dibagi n")
print("             (sandwich estimator via Fisher information)")
print("  • FIC kecil = estimasi kuantil ekor lebih akurat")
print("  • AFIC kecil = performa konsisten di seluruh ekor [Q90-Q99]")
print()

# ============================================================
# 7. RANGKUMAN AKHIR
# ============================================================

print("=" * 65)
print("         RANGKUMAN PERBANDINGAN MODEL")
print("=" * 65)
print()
print(f"  {'Distribusi':<15} {'Log-L':>10}  {'AIC':>9}  {'BIC':>9}  " \
      f"{'KS-D':>8}  {'KS-P':>8}  {'FIC':>14}  {'AFIC':>14}")
print("  " + "-" * 94)
for name, D_val, p_val in ks_results: # Use ks_results for D_val and p_val
    v = results[name]
    fr = fic_records.get(name, {})
    fv   = fr.get('fic',  float('nan'))
    afv  = fr.get('afic', float('nan'))
    print(f"  {name:<15} {v['ll']:>10.4f}  {v['aic']:>9.4f}  " \
          f"{v['bic']:>9.4f}  {D_val:>8.4f}  {p_val:>8.4f}  {fv:>14.8f}  {afv:>14.8f}")

print()
print(f"  Tingkat Signifikansi KS (\u03B1) : {alpha_level:.2f}")
print()
print(f"  Berdasarkan AIC/BIC       : {best_aic}")

# Find best based on KS p-value (highest p-value, or lowest D-stat if p-values are all low)
# For KS, a higher p-value indicates better fit (fail to reject H0)
# Filter out distributions with p-value < alpha_level, then pick the one with the highest p-value
passing_ks_models = [k for k, v in results.items() if v['ks_p'] >= alpha_level]
if passing_ks_models:
    best_ks = max(passing_ks_models, key=lambda nm: results[nm]['ks_p'])
else:
    # If no model passes KS, pick the one with the highest p-value anyway as a 'best'
    best_ks = max(results, key=lambda nm: results[nm]['ks_p'])
print(f"  Berdasarkan KS (p-value > {alpha_level}) : {best_ks}")

print(f"  Berdasarkan FIC           : {best_fic}")
print(f"  Berdasarkan AFIC          : {best_afic}")
print()

# ============================================================
# 8. DD-PLOT DAN HISTOGRAM LENGKAP
# ============================================================

x_sort = np.sort(x) # Add this line back

F_emp = np.arange(1, n+1) / n
ax2.set_title('DD-Plot (Empiris vs Teoritis)', fontsize=12, fontweight='bold')
ax2.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Referensi y=x')

plot_colors = ['tomato', 'seagreen', 'darkorange', 'purple']
for (label, cdf_f), col in zip([
    ('Gamma',        lambda xi: gamma_dist.cdf(xi, a=alpha_mle, scale=1/beta_mle)),
    ('Weibull',      lambda xi: weibull_min.cdf(xi, c=k_wb_mle, scale=lam_wb_mle)),
    ('Lognormal',    lambda xi: lognorm_dist.cdf(xi, s=sigma_ln_mle, scale=np.exp(mu_ln_mle))),
    ('Eksponensial', lambda xi: expon_dist.cdf(xi, scale=1/lambda_exp_mle)),
], plot_colors):
    F_teo = np.array([cdf_f(xi) for xi in x_sort])
    ax2.plot(F_teo, F_emp, 'o-', markersize=3, linewidth=1,
             color=col, label=label, alpha=0.85)

ax2.set_xlabel('CDF Teoritis F(x)')
ax2.set_ylabel('CDF Empiris Fn(x)')
ax2.legend(fontsize=8)
ax2.grid(True, alpha=0.3)

from scipy.stats import gaussian_kde
kde_fit = gaussian_kde(x)
x_line  = np.linspace(x.min(), x.max(), 300)
ax1.plot(x_line, kde_fit(x_line), 'r-', linewidth=1.5, label='KDE')
scale_g = 1 / beta_mle
y_gamma = gamma_dist.pdf(x_line, a=alpha_mle, scale=scale_g)
ax1.plot(x_line, y_gamma, '--', color='seagreen', linewidth=1.5, label='Gamma MLE')
ax1.legend(fontsize=8)

plt.tight_layout()
os.makedirs('/mnt/user-data/outputs/', exist_ok=True) # Create the directory if it doesn't exist
plt.savefig('/mnt/user-data/outputs/grafik_benefit_v2.png', dpi=150, bbox_inches='tight')
plt.close()
print("  [Grafik tersimpan: grafik_benefit_v2.png]")
print()
print("=" * 65)
print("  SELESAI")
print("=" * 65)

         STATISTIKA DESKRIPTIF DEATH BENEFIT
  n (observasi)             : 40
  Mean                      : 217,948.83
  Median                    : 209,142.00
  Modus (approx)            : 10,248
  Variansi                  : 21,129,617,411.84
  Standar Deviasi           : 145,360.30
  Koefisien Variasi (CV)    : 66.69%
  Minimum                   : 10,248
  Maksimum                  : 738,234
  Range                     : 727,986
  Q1 (P25)                  : 138,690.00
  Q2 (P50)                  : 209,142.00
  Q3 (P75)                  : 264,141.00
  IQR                       : 125,451.00
  Skewness                  : 1.6832
  Kurtosis (excess)         : 4.3574
  Kuantil 90%               : 350,352.00
  Kuantil 95%               : 473,646.00
  Kuantil 99%               : 696,088.26

         ESTIMASI PARAMETER MLE

  [1] Distribusi GAMMA (sesuai PDF gambar: beta = rate)
      alpha (shape) = 2.145435
      beta  (rate)  = 9.843754e-06
      E[X] teoritis = 217,948.83
      Log-Like